# DP2 light curve of a single target

**Author:** Mojgan Aghakhanloo (University of Virginia)  
**Runs on:** Rubin Science Platform Notebook aspect, Data Preview 2

Retrieves the forced-photometry light curve of the `DiaObject` at a given position: a cone search on `DiaObject`, then `ForcedSourceOnDiaObject` joined to `Visit` for exposure times. It plots difference-image and direct-image AB magnitudes in all six bands. Set `ra_var` and `dec_var` to any target. This is the single-object building block of the full search in `01_dp2_impostor_search.ipynb`.

*Outputs are cleared because DP2 data are proprietary to Rubin data-rights holders.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u

from lsst.rsp import RSPDiscovery
from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

In [ ]:
discovery = RSPDiscovery("dp2")
service = discovery.get_tap_client()

In [ ]:
filter_colors = get_multiband_plot_colors()
filter_names = filter_colors.keys()
filter_symbols = get_multiband_plot_symbols()

In [ ]:
ra_var = 337.6532356 
dec_var = -0.1101491

In [ ]:
search_rad = 1/3600

In [ ]:
query = "SELECT ra, dec, diaObjectId "\
        "FROM dp2.DiaObject "\
        "WHERE CONTAINS(POINT('ICRS', ra, dec), "\
        "CIRCLE('ICRS', " + str(ra_var) + ", "\
        + str(dec_var) + ", " + str(search_rad) + ")) = 1"
print(query)

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

In [ ]:
assert job.phase == 'COMPLETED'
DiaObj = job.fetch_result().to_table()
print(np.unique(DiaObj['diaObjectId']))

In [ ]:
DiaObjID = DiaObj['diaObjectId'][0]

In [ ]:
query = "SELECT fsodo.diaObjectId, fsodo.visit, fsodo.band, "\
        "fsodo.psfDiffFlux, fsodo.psfDiffFluxErr, "\
        "fsodo.psfFlux as psfFlux, fsodo.psfFluxErr, "\
        "fsodo.psfDiffFlux_flag, fsodo.diff_PixelFlags_nodataCenter, "\
        "fsodo.pixelFlags_saturatedCenter, fsodo.invalidPsfFlag, "\
        "fsodo.pixelFlags_bad, fsodo.psfFlux_flag, "\
        "vis.expMidptMJD "\
        "FROM dp2.ForcedSourceOnDiaObject as fsodo "\
        "JOIN dp2.Visit as vis ON vis.visit = fsodo.visit "\
        "WHERE fsodo.diaObjectId = "+str(DiaObjID)
print(query)

In [ ]:
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
print('Job phase is', job.phase)
if job.phase == 'ERROR':
    job.raise_if_error()

In [ ]:
assert job.phase == 'COMPLETED'
forced_source = job.fetch_result().to_table()
print(len(forced_source))

In [ ]:
for band in filter_names:
    print(f"{band}: {np.sum((forced_source['band'] == band))} visits")

In [ ]:
diff_okflag = (forced_source['psfDiffFlux_flag'] == 0) &\
              (forced_source['diff_PixelFlags_nodataCenter'] == 0) &\
              (forced_source['pixelFlags_saturatedCenter'] == 0) &\
              (forced_source['invalidPsfFlag'] == 0)

science_okflag = (forced_source['psfFlux_flag'] == 0) &\
                 (forced_source['pixelFlags_bad'] == 0) &\
                 (forced_source['pixelFlags_saturatedCenter'] == 0) &\
                 (forced_source['invalidPsfFlag'] == 0)

In [ ]:
# AB magnitude columns (nJy zeropoint = 31.4); non-positive fluxes -> NaN, which simply don't plot
with np.errstate(invalid='ignore', divide='ignore'):
    forced_source['psfMag'] = (np.asarray(forced_source['psfFlux']) * u.nJy).to(u.ABmag).value
    forced_source['psfDiffMag'] = (np.asarray(forced_source['psfDiffFlux']) * u.nJy).to(u.ABmag).value

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
fig.subplots_adjust(wspace=0, hspace=0)

for band in filter_names:
    pickband = (forced_source['band'] == band)
    sel1 = pickband & diff_okflag
    sel2 = pickband & science_okflag

    ax1.plot(forced_source[sel1]['expMidptMJD'],
             forced_source[sel1]['psfDiffMag'],
             marker=filter_symbols[band], color=filter_colors[band],
             label=f"{band}", markersize=7, linestyle='none', fillstyle='none')
    ax2.plot(forced_source[sel2]['expMidptMJD'],
             forced_source[sel2]['psfMag'],
             marker=filter_symbols[band], color=filter_colors[band],
             linestyle='none', markersize=7, fillstyle='none')

ax1.set_ylabel('PSF difference image AB magnitude')
ax2.set_ylabel('PSF direct image AB magnitude')
ax2.set_xlabel('exposure midpoint, MJD (d)')
ax1.invert_yaxis()
ax2.invert_yaxis()
ax1.minorticks_on()
ax2.minorticks_on()
ax1.legend(ncols=2)
plt.show()

## Same result with the `impostorsearch` package

In [ ]:
from impostorsearch.queries import lightcurve_at_position
from impostorsearch.plots import plot_lightcurve

objs, phot = lightcurve_at_position(service, ra_var, dec_var, radius_arcsec=1.0)
for oid in objs["diaObjectId"]:
    plot_lightcurve(phot, oid)
    plt.show()